In [ ]:
# === Common notebook preamble (load llm_math package) ===
import sys
from pathlib import Path

# Candidate paths to locate the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Also add parent directories of the current directory (when running from notebooks/)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] Failed to load the llm_math package: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === End preamble ===


# Ch 02. Matrices and Linear Transformations - Reorganizing Data

> **Learning Goals**
> - Understand matrices as tables of numbers and as linear transformations.
> - Connect matrix multiplication, inverse matrices, eigenvectors, and SVD to code.
> - Understand why matrix multiplication dominates the compute cost of neural networks.

## 2.1 What Is a Matrix?

A matrix is a two-dimensional array of numbers:

$$A \in \mathbb{R}^{m \times n}$$

- $m$: number of rows
- $n$: number of columns
- In data terms: a batch, a feature table, or an image
- In model terms: a weight matrix or a projection matrix

In LLMs, most operations are matrix multiplications. Embedding lookup, linear layers, attention projections, and MLP layers can all be expressed as matrices.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from llm_math.viz import plot_heatmap

# Create matrices
A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)
print(f"A shape: {A.shape}")
print(f"A:\n{A}")
print(f"A^T (Transpose):\n{A.T}")

# Special matrices
I = np.eye(3)  # identity matrix
D = np.diag([1, 2, 3])  # diagonal matrix
print(f"\nIdentity matrix I:\n{I}")
print(f"\nDiagonal matrix D:\n{D}")


## 2.2 Matrix Operations

### Matrix Addition
Two matrices with the same shape are added element by element.

### Matrix Multiplication

$$C = AB, \quad C_{ij} = \sum_k A_{ik}B_{kj}$$

Matrix multiplication is the core operation of neural networks.

Example:
- Input $X \in \mathbb{R}^{B \times d}$
- Weight $W \in \mathbb{R}^{d \times h}$
- Output $Y = XW \in \mathbb{R}^{B \times h}$

In LLMs, $B$ is the number of tokens or batches, $d$ is the embedding dimension, and $h$ is the hidden dimension.


In [ ]:
# Direct matrix multiplication implementation vs np.dot
def matmul_naive(A, B):
    """Triple-loop matrix multiplication implementation for understanding the principle."""
    m, n = A.shape
    n2, p = B.shape
    assert n == n2, f"shape mismatch: {A.shape} vs {B.shape}"
    C = np.zeros((m, p))
    for i in range(m):
        for j in range(p):
            s = 0.0
            for k in range(n):
                s += A[i, k] * B[k, j]
            C[i, j] = s
    return C

A = np.random.randn(100, 100)
B = np.random.randn(100, 100)

# Direct implementation
C_naive = matmul_naive(A, B)
# NumPy
C_numpy = A @ B

# Verify that the results match
print(f"Maximum error: {np.max(np.abs(C_naive - C_numpy)):.2e}")
print(f"Do the two results match? {np.allclose(C_naive, C_numpy)}")


## 2.3 Linear Transformations Through Matrices

A matrix can transform space:

$$\mathbf{y} = A\mathbf{x}$$

Depending on $A$, the transformation can rotate, scale, shear, or project the input. Weight matrices in neural networks learn useful transformations of the input representation.


In [ ]:
# Visualize linear transformations
from llm_math.viz import plot_vector_2d, setup_axes_2d

theta = np.pi / 4  # 45 degrees
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
S = np.array([[2, 0],
              [0, 0.5]])
H = np.array([[1, 1],
              [0, 1]])

# Vertices of the original unit square
square = np.array([[0, 0], [1, 0], [1, 1], [0, 1], [0, 0]]).T  # (2, 5)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Original
ax = axes[0]
ax.plot(square[0], square[1], 'b-', linewidth=2)
ax.fill(square[0], square[1], alpha=0.2, color='blue')
setup_axes_2d(ax, xlim=(-2, 3), ylim=(-2, 3))
ax.set_title('Original')

# Rotation
ax = axes[1]
sq_R = R @ square
ax.plot(sq_R[0], sq_R[1], 'r-', linewidth=2)
ax.fill(sq_R[0], sq_R[1], alpha=0.2, color='red')
setup_axes_2d(ax, xlim=(-2, 3), ylim=(-2, 3))
ax.set_title(f'Rotation (45°)')

# Scaling
ax = axes[2]
sq_S = S @ square
ax.plot(sq_S[0], sq_S[1], 'g-', linewidth=2)
ax.fill(sq_S[0], sq_S[1], alpha=0.2, color='green')
setup_axes_2d(ax, xlim=(-2, 3), ylim=(-2, 3))
ax.set_title('Scaling (2x, 0.5x)')

# Shear
ax = axes[3]
sq_H = H @ square
ax.plot(sq_H[0], sq_H[1], 'm-', linewidth=2)
ax.fill(sq_H[0], sq_H[1], alpha=0.2, color='magenta')
setup_axes_2d(ax, xlim=(-2, 3), ylim=(-2, 3))
ax.set_title('Shear (shear)')

plt.tight_layout()
plt.savefig('../figures/ch02_linear_transforms.png', dpi=100, bbox_inches='tight')
plt.show()
print("Each matrix transforms space. Every weight matrix in a transformer is also a linear transformation.")


## 2.4 Inverse Matrices and Their Meaning

If a square matrix $A$ has an inverse $A^{-1}$, then:

$$AA^{-1} = A^{-1}A = I$$

The inverse matrix reverses the transformation. If the determinant is 0, the matrix is singular and has no inverse. This means information has been collapsed into a lower-dimensional space.


In [ ]:
# Inverse matrix example
A = np.array([[2, 1],
              [1, 3]], dtype=float)
A_inv = np.linalg.inv(A)
print(f"A:\n{A}")
print(f"\nA^-1:\n{A_inv}")
print(f"\nA @ A^-1 (should be the identity matrix):\n{A @ A_inv}")

# Determinant
det_A = np.linalg.det(A)
print(f"\ndet(A) = {det_A:.4f}")

# Singular matrix example (no inverse)
B = np.array([[1, 2],
              [2, 4]], dtype=float)  # The second row is twice the first row
det_B = np.linalg.det(B)
print(f"\nB (singular matrix):\n{B}")
print(f"det(B) = {det_B:.6f} (zero means no inverse)")


## 2.5 Eigenvalues, Eigenvectors, and Diagonalization

An eigenvector is a vector whose direction does not change under a linear transformation:

$$A\mathbf{v} = \lambda \mathbf{v}$$

- $\mathbf{v}$: eigenvector
- $\lambda$: eigenvalue

Eigenvectors reveal the principal directions of a transformation. This idea is connected to PCA, SVD, and low-rank approximation.


In [ ]:
# Eigenvalues/eigenvectors
A = np.array([[4, -2],
              [1,  1]], dtype=float)
eigenvalues, eigenvectors = np.linalg.eig(A)
print(f"A:\n{A}")
print(f"\nEigenvalues: {eigenvalues}")
print(f"Eigenvector:\n{eigenvectors}")

# Verify: A v = lambda v
for i, (lam, v) in enumerate(zip(eigenvalues, eigenvectors.T)):
    Av = A @ v
    lv = lam * v
    print(f"\nEigenvector {i+1}: v = {v}, lambda = {lam:.4f}")
    print(f"  A @ v = {Av}")
    print(f"  lambda * v = {lv}")
    print(f"  Match? {np.allclose(Av, lv)}")


In [ ]:
# SVD-based image compression (low-rank approximation)
from sklearn.datasets import load_digits

X_train, y_train = load_digits(return_X_y=True)
print(f"MNIST data: {X_train.shape}")

# Reshape the first image to 28x28
img = X_train[0].reshape(8, 8)

# SVD
U, s, Vt = np.linalg.svd(img, full_matrices=False)
print(f"U: {U.shape}, s: {s.shape}, Vt: {Vt.shape}")
print(f"Singular values (top 10): {s[:10]}")

# Rank-k approximation
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original (rank 8)')
axes[0].axis('off')

for i, k in enumerate([1, 2, 4, 8]):
    img_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    axes[i+1].imshow(img_k, cmap='gray')
    axes[i+1].set_title(f'rank-{k} approx.')
    axes[i+1].axis('off')

plt.tight_layout()
plt.savefig('../figures/ch02_svd_compression.png', dpi=100, bbox_inches='tight')
plt.show()
print("\n=> Even a rank-10 approximation preserves the digit shape. SVD extracts the core information in the data.")


## 2.6 [CPU/GPU Benchmark 1] Matrix Multiplication by Size

Matrix multiplication has $O(n^3)$ complexity. GPUs greatly accelerate this operation through parallel computation.


In [ ]:
# CPU vs GPU benchmark by matrix multiplication size
import time
import numpy as np
from llm_math.bench import time_fn, format_results_table

try:
    import torch
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("PyTorch is unavailable, so only NumPy CPU is measured.")

# Matrix sizes to test
sizes = [64, 256, 512, 1024, 2048]

results = {}
print(f"{'n':>6} | {'NumPy CPU (ms)':>16} | {'Torch CPU (ms)':>16} | {'Torch GPU (ms)':>16} | {'Speedup (GPU/CPU)':>18}")
print("-" * 80)

for n in sizes:
    A_np = np.random.randn(n, n).astype(np.float32)
    B_np = np.random.randn(n, n).astype(np.float32)

    # NumPy CPU
    def np_matmul(A, B):
        return A @ B
    t_np = time_fn(np_matmul, A_np, B_np, device='cpu', warmup=2, repeat=3)['mean_ms']

    if HAS_TORCH:
        A_t = torch.from_numpy(A_np)
        B_t = torch.from_numpy(B_np)
        def t_matmul(A, B):
            return A @ B
        t_torch_cpu = time_fn(t_matmul, A_t, B_t, device='cpu', warmup=2, repeat=3)['mean_ms']

        if torch.cuda.is_available():
            A_gpu = A_t.cuda()
            B_gpu = B_t.cuda()
            t_torch_gpu = time_fn(t_matmul, A_gpu, B_gpu, device='cuda', warmup=3, repeat=5)['mean_ms']
            speedup = t_torch_cpu / t_torch_gpu if t_torch_gpu > 0 else float('inf')
            print(f"{n:>6} | {t_np:>16.3f} | {t_torch_cpu:>16.3f} | {t_torch_gpu:>16.3f} | {speedup:>17.2f}x")
        else:
            print(f"{n:>6} | {t_np:>16.3f} | {t_torch_cpu:>16.3f} | {'N/A':>16} | {'N/A':>18}")
    else:
        print(f"{n:>6} | {t_np:>16.3f} | {'N/A':>16} | {'N/A':>16} | {'N/A':>18}")

if not (HAS_TORCH and torch.cuda.is_available()):
    print("\n⚠ No GPU was detected. In Colab, switch the runtime to a T4 GPU to see acceleration.")
    print("  Runtime → Change runtime type → T4 GPU")


In [ ]:
# Visualization: compare time by matrix size (log scale)
import matplotlib.pyplot as plt

# Recollect results from the previous cell so this cell can run independently
results_collect = {'n': [], 'numpy_cpu': [], 'torch_cpu': [], 'torch_gpu': []}
for n in [64, 256, 512, 1024, 2048]:
    A_np = np.random.randn(n, n).astype(np.float32)
    B_np = np.random.randn(n, n).astype(np.float32)
    t_np = time_fn(lambda A, B: A @ B, A_np, B_np, device='cpu', warmup=1, repeat=2)['mean_ms']
    results_collect['n'].append(n)
    results_collect['numpy_cpu'].append(t_np)
    if HAS_TORCH:
        A_t = torch.from_numpy(A_np); B_t = torch.from_numpy(B_np)
        t_tc = time_fn(lambda A, B: A @ B, A_t, B_t, device='cpu', warmup=1, repeat=2)['mean_ms']
        results_collect['torch_cpu'].append(t_tc)
        if torch.cuda.is_available():
            A_g = A_t.cuda(); B_g = B_t.cuda()
            t_tg = time_fn(lambda A, B: A @ B, A_g, B_g, device='cuda', warmup=2, repeat=3)['mean_ms']
            results_collect['torch_gpu'].append(t_tg)
        else:
            results_collect['torch_gpu'].append(None)
    else:
        results_collect['torch_cpu'].append(None)
        results_collect['torch_gpu'].append(None)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(results_collect['n'], results_collect['numpy_cpu'], 'o-', label='NumPy CPU')
if results_collect['torch_cpu'][0] is not None:
    ax.plot(results_collect['n'], results_collect['torch_cpu'], 's-', label='PyTorch CPU')
if results_collect['torch_gpu'][0] is not None:
    ax.plot(results_collect['n'], results_collect['torch_gpu'], '^-', label='PyTorch GPU', linewidth=2)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Matrix size n')
ax.set_ylabel('Time (ms)')
ax.set_title('Matrix multiplication speed comparison: the world of $O(n^3)$')
ax.legend()
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch02_bench_matmul.png', dpi=100, bbox_inches='tight')
plt.show()
print("On the log-log plot, a slope near 3 matches the $O(n^3)$ prediction.")


## 2.7 Key Takeaways

| Concept | Formula | Meaning |
|---|---|---|
| Matrix multiplication | $C_{ij}=\sum_k A_{ik}B_{kj}$ | Linear combination |
| Linear transformation | $\mathbf{y}=A\mathbf{x}$ | Transform space |
| Inverse matrix | $AA^{-1}=I$ | Reverse a transformation |
| Eigenvector | $A\mathbf{v}=\lambda\mathbf{v}$ | Direction preserved by transformation |
| SVD | $A=U\Sigma V^T$ | Low-rank structure |

### Practice Problems
1. Multiply two $2\times2$ matrices by hand and verify with NumPy.
2. Visualize how a rotation matrix transforms a vector.
3. Create a singular matrix and explain why it has no inverse.
4. Compute the eigenvalues and eigenvectors of a symmetric matrix.
5. Approximate an image with SVD using ranks 1, 5, and 10, and compare the visual quality.
